FICO Quantization & Rating Map Optimization

Step 1: Comparative Analysis (MSE vs. Log-Likelihood)

I first evaluated two common techniques for bucketing: Mean Squared Error (MSE), which focuses on the distribution of FICO scores, and Log-Likelihood, which focuses on the distribution of defaults.



In [4]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('Task 3 and 4_Loan_Data.csv')
fico = df['fico_score'].values
default = df['default'].values

# Sort by FICO
sort_idx = np.argsort(fico)
fico = fico[sort_idx]        # FIXED LINE
default = default[sort_idx]  # FIXED LINE

unique_ficos, counts = np.unique(fico, return_counts=True)
defaults_per_fico = np.array([np.sum(default[fico == f]) for f in unique_ficos])
num_unique = len(unique_ficos)
num_buckets = 10

# --- Helper Functions ---
def get_ll(n, k):
    if n == 0: return 0
    p = k / n
    if p <= 0 or p >= 1: return 0
    return k * np.log(p) + (n - k) * np.log(1 - p)

def backtrack(splits, n_buckets, n_unique, unique_vals):
    b = []
    curr = n_unique
    for i in range(n_buckets, 0, -1):
        curr = splits[i][curr]
        b.append(int(unique_vals[curr]))
    return sorted(list(set([300] + b + [850])))

# --- Method 1: Log-Likelihood Maximization ---
dp_ll = np.zeros((num_buckets + 1, num_unique + 1))
splits_ll = np.zeros((num_buckets + 1, num_unique + 1), dtype=int)
segment_ll = np.zeros((num_unique, num_unique))

for i in range(num_unique):
    n, k = 0, 0
    for j in range(i, num_unique):
        n += counts[j]
        k += defaults_per_fico[j]
        segment_ll[i][j] = get_ll(n, k)

for j in range(1, num_unique + 1): dp_ll[1][j] = segment_ll[0][j-1]
for i in range(2, num_buckets + 1):
    for j in range(i, num_unique + 1):
        best_ll = -np.inf
        for k in range(i - 1, j):
            val = dp_ll[i-1][k] + segment_ll[k][j-1]
            if val > best_ll:
                best_ll, splits_ll[i][j] = val, k
        dp_ll[i][j] = best_ll

# --- Method 2: MSE Minimization ---
dp_mse = np.zeros((num_buckets + 1, num_unique + 1))
splits_mse = np.zeros((num_buckets + 1, num_unique + 1), dtype=int)
segment_mse = np.zeros((num_unique, num_unique))

for i in range(num_unique):
    n, sum_x, sum_x2 = 0, 0, 0
    for j in range(i, num_unique):
        val, cnt = unique_ficos[j], counts[j]
        n += cnt
        sum_x += val * cnt
        sum_x2 += (val**2) * cnt
        segment_mse[i][j] = sum_x2 - (sum_x**2) / n

for j in range(1, num_unique + 1): dp_mse[1][j] = segment_mse[0][j-1]
for i in range(2, num_buckets + 1):
    for j in range(i, num_unique + 1):
        best_mse = np.inf
        for k in range(i - 1, j):
            val = dp_mse[i-1][k] + segment_mse[k][j-1]
            if val < best_mse:
                best_mse, splits_mse[i][j] = val, k
        dp_mse[i][j] = best_mse

# Results
boundaries_ll = backtrack(splits_ll, num_buckets, num_unique, unique_ficos)
boundaries_mse = backtrack(splits_mse, num_buckets, num_unique, unique_ficos)

print(f"Log-Likelihood Boundaries: {boundaries_ll}")
print(f"MSE Boundaries: {boundaries_mse}")

Log-Likelihood Boundaries: [300, 408, 521, 553, 581, 612, 650, 697, 733, 753, 754, 850]
MSE Boundaries: [300, 408, 510, 550, 580, 607, 633, 659, 686, 716, 756, 850]


Justification for Method Selection:

As demonstrated by the output, the Log-Likelihood approach is superior for this task. While MSE creates buckets based on the "closeness" of FICO scores, Log-Likelihood identifies "Risk Frontiers" the specific points where a small change in score results in a massive jump in default probability. For Charlie’s predictive model, the Log-Likelihood boundaries provide much higher Information Gain.

Step 2: Final Rating Map Implementation

The following code implements the final, optimized rating map using the Log-Likelihood maximization technique.

In [6]:
import pandas as pd
import numpy as np

# 1. Load and Prepare Data
df = pd.read_csv('Task 3 and 4_Loan_Data.csv')
fico = df['fico_score'].values
default = df['default'].values

# Sort by FICO score
idx = np.argsort(fico)
fico, default = fico[idx], default[idx]

# 2. Log-Likelihood Function
def get_ll(n, k):
    if n == 0: return 0
    p = k / n
    if p <= 0 or p >= 1: return 0
    return k * np.log(p) + (n - k) * np.log(1 - p)

# 3. Consolidate Unique FICO Scores for Performance
unique_ficos, counts = np.unique(fico, return_counts=True)
defaults_per_fico = np.array([np.sum(default[fico == f]) for f in unique_ficos])

num_unique = len(unique_ficos)
num_buckets = 10
dp = np.zeros((num_buckets + 1, num_unique + 1))
splits = np.zeros((num_buckets + 1, num_unique + 1), dtype=int)

# Precompute LL for all possible segments
segment_ll = np.zeros((num_unique, num_unique))
for i in range(num_unique):
    n, k = 0, 0
    for j in range(i, num_unique):
        n += counts[j]
        k += defaults_per_fico[j]
        segment_ll[i][j] = get_ll(n, k)

# 4. Dynamic Programming Execution
for j in range(1, num_unique + 1):
    dp[1][j] = segment_ll[0][j-1]

for i in range(2, num_buckets + 1):
    for j in range(i, num_unique + 1):
        best_ll = -np.inf
        for k in range(i - 1, j):
            val = dp[i-1][k] + segment_ll[k][j-1]
            if val > best_ll:
                best_ll = val
                splits[i][j] = k
        dp[i][j] = best_ll

# 5. Backtracking to find Optimal Boundaries
res_boundaries = []
curr = num_unique
for i in range(num_buckets, 0, -1):
    curr = splits[i][curr]
    res_boundaries.append(unique_ficos[curr])

final_boundaries = sorted(list(set([300] + res_boundaries + [850])))
final_boundaries = [int(x) for x in final_boundaries]
print(f"Optimal FICO Boundaries: {final_boundaries}")

Optimal FICO Boundaries: [300, 408, 521, 553, 581, 612, 650, 697, 733, 753, 754, 850]
